
# 07 期货交易策略：核心笔记

> **本章要解决的问题**：期货策略为什么先讲“波动率目标”和“头寸规模”，而不是先讲信号？趋势、carry、均值回归、相对价值各自赚什么钱？为什么同一套策略，资金少的人做不了、成本高的人做不赚？

## 本章导读

这一章是 Robert Carver《Advanced Futures Trading Strategies》的零基础笔记。Carver 的核心观点很朴素：**先把风险管好，再谈信号；先算成本，再谈收益。**

按“基础 → 基础策略 → 进阶策略 → 战术 → 对普通人的用法”展开：

1. 期货基础：合约、保证金、乘数、展期、连续合约、波动率目标、头寸规模；
2. Part One 基础方向策略：买入持有、风险缩放、慢/快趋势、趋势强度、多条趋势规则、carry、趋势+carry；
3. Part Two 进阶趋势/carry：调整/标准化趋势、风险 regime、现货趋势、精确 carry、趋势/carry 配置、资产类别、截面动量/截面 carry；
4. Part Three 进阶方向：突破、价值、加速、偏度、动态优化；
5. Part Four 快速均值回归；Part Five 相对价值：跨品种价差、三腿、日历；
6. Part Six 战术：合约选择/展期、执行、现金/复利、风险管理、成本与回测陷阱；
7. 对一人量化的用法：先波动率目标与风险预算，再 trend/carry，最后碰相对价值；不要一次上 30 个策略。

**重要说明**：本章所有品种、价格、carry、合约参数都是**合成教学数据**，为了教学把“可预测结构”放大了；真实市场没有这么友好，数字不能直接拿去交易。

### 学习目标

- 能说清期货合约的基本要素，以及“连续合约”和展期收益（roll yield）的关系。
- 能用“目标波动 / 实现波动”算仓位，并解释风险缩放。
- 能说出慢趋势、快趋势、多窗口趋势、carry 的区别与组合逻辑。
- 能跑通一个 8 品种、1000 天的期货组合回测，并算出收益、波动、回撤、换手。
- 能用成本敏感性说明“换手高为什么致命”。
- 能解释为什么散户需要足够资本（约 10 万美元级只是门槛）。

### 先修

会基础统计（平均、标准差、相关系数）；会一点 Python（numpy/pandas）。不需要期货账户或交易经验。

---

## 在量化研究中的位置

- **谁用**：CTA/期货策略研究员、商品交易员、资产配置、一人量化。
- **输入**：合约规则、连续价格、期限结构、波动率、成本与资本。
- **输出**：波动率目标仓位、组合权重、回测曲线、成本/容量评估。
- **在流程的哪一步**：在“信号研究”之后、“实盘执行”之前；它把“信号”翻译成“头寸与风险”。
- **失败模式**：跳过波动率目标直接上信号、忽略展期、成本算少、杠杆过大、用 30 个策略堆在一起、回测按收盘价成交。
- **最小工作流**：合约与数据 → 波动率目标 → 趋势/carry 信号 → 风险预算组合 → 成本与容量 → 回测与压力测试。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 期货基础：合约、展期与波动率目标

### 1.1 一张期货合约看什么

| 要素 | 说明 | 为什么重要 |
|---|---|---|
| 标的 | 品种与交割月 | 决定价格跟谁走 |
| 合约乘数 | 每点/每吨值多少钱 | 换算名义价值与盈亏 |
| 最小变动 | 一跳多少钱 | 报价与成本精度 |
| 保证金 | 开仓要压多少钱 | 杠杆与强平风险 |
| 到期/交割 | 合约有寿命 | 必须换月 |

### 1.2 连续合约与展期（roll）

期货会到期，策略要长期持有就必须**换月**：卖出近月、买入远月。不同月份价格不同，换月会产生**展期收益（roll yield）**：远月比近月便宜（贴水/backwardation）时，多头换月“越换越便宜”，有正 carry；反之是负 carry。

所以期货的“总收益”可以拆成：

$$\text{总收益}\approx\text{现货价格变化}+\text{展期收益}.$$

**忽略展期，回测会系统性偏差。** 本章代码会把这两块拆开对比。

### 1.3 波动率目标与风险缩放

Carver 的第一原则：**先决定每个品种承担多少风险，再决定买多少手。**

- 目标波动：整个组合年化波动目标，例如 10%；
- 风险缩放：单个品种的仓位 $\propto 1/\text{该品种波动}$；
- 波动率目标：仓位再乘 $\text{目标波动}/\text{实现波动}$，波动上升自动减仓。

$$\text{权重的风险贡献}=\text{权重}\times\text{品种波动}\approx\text{目标风险}.$$

### 1.4 为什么散户需要足够资本

一手合约的名义价值可能几十万，而 10% 目标波动下每个品种只能承担很小风险。资金少时，会出现两种尴尬：

- 算出来的仓位不足 1 手，只能开 0 手 → 策略没法执行；
- 为了凑够 1 手，实际风险远超目标 → 集中、爆仓风险大。

Carver 提到的“约 10 万美元级”只是**最低门槛**；品种越多、合约越大，需要的资本越多。

---

## 2. 数据：8 个品种、1000 天与展期影响

### 2.1 代码：合成面板与展期分解



In [2]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(2307)
N, T = 8, 1000
names = ["EQ1", "EQ2", "BD1", "BD2", "MT1", "MT2", "AG1", "EN1"]
asset_class = ["股指", "股指", "国债", "国债", "金属", "金属", "农产品", "能源"]
price0 = np.array([4500, 3800, 120, 105, 68000, 22000, 3800, 520], dtype=float)
sig_d = np.array([0.012, 0.010, 0.004, 0.003, 0.013, 0.011, 0.012, 0.016])
beta = np.array([1.0, 0.9, 0.2, 0.15, 0.6, 0.5, 0.3, 1.1])

# 1) 潜在趋势状态（慢变量）+ carry（年化，AR(1)）
theta = np.zeros((T, N))
theta[0] = rng.normal(0, 1, N)
carry = np.zeros((T, N))
carry[0] = rng.uniform(-0.04, 0.05, N)
for t in range(1, T):
    theta[t] = 0.99 * theta[t - 1] + 0.02 * rng.normal(0, 1, N)
    carry[t] = 0.97 * carry[t - 1] + rng.normal(0, 0.01, N)

# 2) 日收益 = 市场 + 趋势漂移 + carry/252 + 噪声；去掉样本均值便于对比策略
mkt = rng.normal(0, 0.009, T)
daily_carry = carry / 252
ret = (beta[None, :] * mkt[:, None] + 0.004 * theta + daily_carry
       + sig_d[None, :] * rng.normal(0, 1, (T, N)))
ret = ret - ret.mean(axis=0, keepdims=True)
price = np.zeros((T, N))
price[0] = price0
for t in range(1, T):
    price[t] = price[t - 1] * (1 + ret[t])

# 3) 期限结构：近月 = 价格，远月 = 价格 * exp(-carry * 60/365)
far = price * np.exp(-carry * 60 / 365)
carry_obs = np.log(price / far) * 365 / 60

print("面板：", price.shape, "（天数, 品种）；合成教学数据")
print(f"{'品种':<5}{'类别':<4}{'起始价':>9}{'年化波动':>9}{'平均carry':>10}")
for i in range(N):
    print(f"{names[i]:<5}{asset_class[i]:<4}{price0[i]:>9.0f}"
          f"{ret[:, i].std(ddof=1) * np.sqrt(252):>9.2%}{carry[:, i].mean():>10.2%}")

# 4) 展期的影响：总收益（含 carry） vs 只看现货价格变化
spot_ret = ret - daily_carry
print("\n展期/持有收益的影响（年化）：")
print(f"{'品种':<5}{'总收益(含carry)':>14}{'现货变化':>10}{'carry贡献':>10}")
for i in range(N):
    print(f"{names[i]:<5}{ret[:, i].mean() * 252:>14.2%}{spot_ret[:, i].mean() * 252:>10.2%}"
          f"{carry[:, i].mean():>10.2%}")


面板： (1000, 8) （天数, 品种）；合成教学数据
品种   类别        起始价     年化波动   平均carry
EQ1  股指       4500   23.76%    -1.55%
EQ2  股指       3800   20.85%    -0.99%
BD1  国债        120    7.23%     1.03%
BD2  国债        105    5.34%     2.53%
MT1  金属      68000   21.78%     0.22%
MT2  金属      22000   19.26%     1.38%
AG1  农产品      3800   19.54%    -1.67%
EN1  能源        520   31.34%    -1.59%

展期/持有收益的影响（年化）：
品种      总收益(含carry)      现货变化   carry贡献
EQ1          -0.00%     1.55%    -1.55%
EQ2           0.00%     0.99%    -0.99%
BD1           0.00%    -1.03%     1.03%
BD2          -0.00%    -2.53%     2.53%
MT1           0.00%    -0.22%     0.22%
MT2          -0.00%    -1.38%     1.38%
AG1           0.00%     1.67%    -1.67%
EN1          -0.00%     1.59%    -1.59%



**读数与应用**：8 个品种年化波动从 5%（国债）到 31%（能源）不等；总收益 = 现货变化 + carry 这一恒等式在表里看得一清二楚：carry 为负的品种，总收益比现货变化更低（多头还要倒贴展期成本）。**做期货策略，第一步不是算信号，而是把“展期”这笔账拆清楚。**

**常见错误**：

1. 用不换月的近月价格做多年回测，忽略展期。
2. 把 carry 当“额外收益”，忘了它对多头是成本、对空头是收益。
3. 用名义价格算波动，忽略不同品种的乘数与单位。

---

## 3. Part One：基础方向策略

### 3.1 买入持有与风险缩放

最朴素的策略是买入持有；但“等金额买”会让高波动品种主导风险。Carver 的做法是**等风险**：每个品种按 $1/\sigma$ 配置，让风险贡献相等。这只是基线，不是 alpha。

### 3.2 慢趋势与快趋势

趋势策略的核心是**移动平均差（EWMAC）**：快均线减慢均线，再按波动标准化。

- 快趋势（如 8/32 天）：反应快、换手高、噪声大；
- 慢趋势（如 32/128、64/256 天）：反应慢、换手低、更稳；
- **多窗口趋势**：把 4 个速度的信号平均，牺牲一点收益换稳定性。

### 3.3 趋势强度、突破与多条规则

趋势强度衡量“趋势有多顺”；突破策略在价格创 N 日新高/新低时入场；多条规则平均可以降低“单一规则失效”的风险。Carver 的结论是：**多规则、多速度平均，比押注单一最优规则更稳。**

### 3.4 carry 与 trend+carry

carry 赚的是“期限结构”的钱，趋势赚的是“价格延续”的钱，两者相关性低、逻辑不同，组合起来通常能提高风险调整后收益。这也是 Carver 反复强调的：**不同风格的低相关组合，比单一策略更值钱。**

---

## 4. 信号：波动率、趋势与 carry

### 4.1 代码：三类信号



In [3]:

def ewma(x, span):
    return pd.DataFrame(x).ewm(span=span, adjust=False).mean().to_numpy()


def rolling_z(a, win=250):
    s = pd.DataFrame(a)
    m = s.rolling(win, min_periods=60).mean()
    v = s.rolling(win, min_periods=60).std()
    return ((s - m) / v).to_numpy()


logp = np.log(price)
vol_36 = pd.DataFrame(ret).rolling(36, min_periods=20).std().to_numpy() * np.sqrt(252)
vol_36 = np.nan_to_num(vol_36, nan=0.12)

trend_speeds = [(8, 32), (16, 64), (32, 128), (64, 256)]
trend_list = []
for fast, slow in trend_speeds:
    raw = ewma(logp, fast) - ewma(logp, slow)
    raw = raw / np.nan_to_num(vol_36, nan=0.12)
    z = rolling_z(raw)
    trend_list.append(np.tanh(np.nan_to_num(z) / 2))
trend = np.nanmean(trend_list, axis=0)
carry_fc = np.tanh(np.nan_to_num(rolling_z(carry_obs)) / 2)

print("趋势信号均值/绝对值均值:", round(float(np.nanmean(trend[100:])), 3),
      round(float(np.nanmean(np.abs(trend[100:]))), 3))
print("carry 信号均值/绝对值均值:", round(float(np.nanmean(carry_fc[100:])), 3),
      round(float(np.nanmean(np.abs(carry_fc[100:]))), 3))
print("趋势与 carry 相关性:", round(float(np.corrcoef(trend[100:].ravel(), carry_fc[100:].ravel())[0, 1]), 3))


趋势信号均值/绝对值均值: -0.022 0.37
carry 信号均值/绝对值均值: 0.007 0.39
趋势与 carry 相关性: 0.066



**读数与应用**：趋势信号绝对值均值约 0.37，carry 信号约 0.39，说明两个信号都“有方向感”，但方向经常相反；趋势与 carry 相关性只有 0.066——这正是它们能互补的原因。信号被限制在 $[-1,1]$，避免极端值让某个品种权重失控。

**常见错误**：

1. 未按波动标准化，高波动品种信号压过其他品种。
2. 用未来的均值/方差做 z-score（未来函数）。
3. 把快趋势当“更好”：换手和成本会吃掉大部分收益。

---

## 5. 组合：波动率目标 + 风险预算 + 成本

### 5.1 从信号到权重

1. 每个品种的目标风险贡献相等：仓位 $\propto \text{信号}/\sigma$；
2. 用历史信号幅度估计“1 单位风险预算”对应多少仓位，使组合目标波动 = 10%；
3. 每天按新信号调仓（Carver 默认日频），换手就是成本。

### 5.2 代码：组合回测与成本敏感性



In [4]:

TARGET_VOL = 0.10
forecast = 0.5 * trend + 0.5 * carry_fc
f2 = np.nansum(forecast ** 2, axis=1)
f2_avg = pd.Series(f2).rolling(250, min_periods=60).mean().shift(1).to_numpy()
f2a = np.nan_to_num(f2_avg)
k = np.where(f2a > 1e-8, TARGET_VOL / np.sqrt(np.maximum(f2a, 1e-8)), 0.0)


def weights_from(fc, kk):
    return np.nan_to_num(kk[:, None] * fc / np.clip(vol_36, 0.03, None))


def stats(pnl):
    pnl = pnl[100:]
    nav = np.cumprod(1 + pnl)
    dd = nav / np.maximum.accumulate(nav) - 1
    return dict(ann=pnl.mean() * 252, vol=pnl.std(ddof=1) * np.sqrt(252),
                sharpe=pnl.mean() / pnl.std(ddof=1) * np.sqrt(252), mdd=dd.min())


W = weights_from(forecast, k)
turn = np.nansum(np.abs(np.diff(W, axis=0)), axis=1)
pnl_gross = np.nansum(W[:-1] * ret[1:], axis=1)
print("组合（趋势+carry 50/50，目标波动 10%）：")
for cb in [0, 3, 10, 20]:
    s = stats(pnl_gross - turn * cb / 1e4)
    print(f"  成本 {cb:>2}bp：年化 {s['ann']:.2%}，波动 {s['vol']:.2%}，"
          f"Sharpe {s['sharpe']:.2f}，最大回撤 {s['mdd']:.2%}")
print(f"  日均换手 {turn[100:].mean():.2f}，年化换手 {turn[100:].mean() * 252:.0f} 倍")


组合（趋势+carry 50/50，目标波动 10%）：
  成本  0bp：年化 16.18%，波动 12.41%，Sharpe 1.30，最大回撤 -27.15%
  成本  3bp：年化 13.02%，波动 12.42%，Sharpe 1.05，最大回撤 -28.92%
  成本 10bp：年化 5.63%，波动 12.45%，Sharpe 0.45，最大回撤 -33.16%
  成本 20bp：年化 -4.91%，波动 12.49%，Sharpe -0.39，最大回撤 -40.26%
  日均换手 0.42，年化换手 105 倍



**读数与应用**：毛 Sharpe 1.30；3bp 成本后 1.05；10bp 成本后只剩 0.45；20bp 成本后转负（−0.39）。年化换手约 105 倍——**日频调仓的期货策略，成本就是生死线。** 另外实际波动 12.4% 高于 10% 目标，因为估计里假设了品种间低相关，实际它们共享市场因子。

**常见错误**：

1. 成本只算手续费，不算滑点与冲击。
2. 用“目标波动”直接当“实际波动”，不复盘偏差。
3. 信号变了就调仓，不考虑最小调仓阈值，换手爆炸。

---

## 6. Part Two/Three 概览与策略对比

### 6.1 进阶策略都在解决什么

- **调整/标准化趋势**：不同品种波动差异大，必须标准化后再比较；
- **风险 regime**：市场波动高时整体降杠杆；
- **现货趋势**：用现货/近月价做趋势，减少换月噪声；
- **精确 carry**：考虑真实展期成本、季节性；
- **截面动量/carry**：在品种间排序做多空，而不是每个品种单独择时；
- **突破/价值/加速/偏度**：不同“趋势的变体”，本质都是在捕捉价格延续或反转的不同侧面；
- **动态优化**：当资金不够交易“所有策略”时，用优化把有限的仓位分配给最有效的组合。

### 6.2 代码：策略对比



In [5]:

port_vol = pd.DataFrame(ret).rolling(60).std().mean(axis=1).to_numpy() * np.sqrt(252)
regime_med = pd.Series(port_vol).rolling(250, min_periods=60).median().shift(1).to_numpy()
regime_scale = np.where(port_vol > np.nan_to_num(regime_med, nan=99), 0.5, 1.0)
k_bh = np.full(T, TARGET_VOL / np.sqrt(N))

strategies = {
    "买入持有(等风险)": weights_from(np.ones((T, N)), k_bh),
    "仅 carry": weights_from(carry_fc, k),
    "快趋势": weights_from(trend_list[0], k),
    "慢趋势": weights_from(trend_list[2], k),
    "多窗口趋势": weights_from(trend, k),
    "趋势+carry": weights_from(forecast, k),
    "趋势+carry+regime": weights_from(forecast, k) * regime_scale[:, None],
}
print(f"{'策略':<16}{'年化':>8}{'波动':>8}{'Sharpe':>8}{'最大回撤':>9}{'换手':>7}")
for name, Wx in strategies.items():
    tx = np.nansum(np.abs(np.diff(Wx, axis=0)), axis=1)
    px = np.nansum(Wx[:-1] * ret[1:], axis=1) - tx * 3 / 1e4
    s = stats(px)
    print(f"{name:<16}{s['ann']:>8.2%}{s['vol']:>8.2%}{s['sharpe']:>8.2f}"
          f"{s['mdd']:>9.2%}{tx[100:].mean():>7.2f}")


策略                    年化      波动  Sharpe     最大回撤     换手
买入持有(等风险)         -0.39%  15.79%   -0.02  -26.86%   0.04
仅 carry            5.76%  16.58%    0.35  -41.27%   0.81
快趋势                7.61%  19.84%    0.38  -40.04%   0.32
慢趋势               23.06%  20.38%    1.13  -28.67%   0.18
多窗口趋势             19.06%  17.30%    1.10  -25.22%   0.19
趋势+carry          13.02%  12.42%    1.05  -28.92%   0.42
趋势+carry+regime    9.26%  10.04%    0.92  -24.82%   0.38



**读数与应用**：买入持有（等风险）年化 −0.39%（数据去均值后只剩噪声）；仅 carry 5.76%、Sharpe 0.35；快趋势 7.61%、0.38；慢趋势 23.06%、1.13；多窗口趋势 19.06%、1.10；趋势+carry 13.02%、Sharpe 1.05，但波动只有 12.4%（比慢趋势的 20.4% 低很多）。regime 过滤把波动降到 10.0%，Sharpe 反而略降——**进阶技巧不一定都提高收益，要一个一个验证。**

**常见错误**：

1. 把多窗口平均当“一定更好”：本例它比慢趋势低。
2. 只看收益：趋势+carry 收益低于慢趋势，但风险低得多。
3. 默认 regime 有效，不做事后验证。

---

### 6.3 原书其他策略：skew、动态优化与截面

- **Skew（策略 24）**：研究收益分布的偏度，把它当作策略/风险特征，而不是单独的方向信号；偏度策略通常容量小、尾部风险大，原书是“案例研究”而非主推。
- **动态优化（策略 25）**：当资金不足以交易“Jumbo 组合”的全部策略时，用优化/风险预算把有限仓位分配给最有效的组合，而不是硬上 30 个策略。
- **截面动量/截面 carry**：不是每个品种单独择时，而是在品种间按动量/carry 排序做多空；与时间序列 trend/carry 是两种不同玩法，容量、换手、相关性都不同。

本章按“策略家族”归纳，不逐条复刻 30 个策略；每个策略的具体规则、参数与回测在原书。

## 7. Part Four/Five 概览：均值回归与相对价值

### 7.1 快速均值回归

短期反转策略（比如 1–5 天）赚“过度反应”的钱，但**对成本极其敏感**：换手高、单笔利润小，成本稍微高一点就由正转负。Carver 的“更安全版本”是加过滤器（只在特定波动/流动性条件下交易）或放慢速度。

### 7.2 相对价值：跨品种价差、三腿、日历

- **跨品种价差**：两个相关品种的价差回归（如两种金属、两种股指）；
- **三腿**：三个品种的组合价差，更复杂、容量更低；
- **日历价差**：同一品种近远月价差，与 carry/库存直接相关。

相对价值策略通常**杠杆更高、容量更小、对执行更敏感**，而且价差可能在结构变化时“一去不回”。Carver 建议：**先把方向性策略做扎实，再碰相对价值。**

---

## 8. Part Six 战术：合约、资本与回测陷阱

### 8.1 合约选择与展期

- 选流动性最好的合约（主力），按规则换月；
- 换月成本要计入回测；不同换月规则会改变收益；
- 用“复权/连续合约”而不是直接拼价格。

### 8.2 资本与整数手

波动率目标算出来的是“名义仓位”，落地要变成整数手。资金少时，整数化会带来两个问题：**仓位不足**（很多品种开不出）和**风险偏差**（凑 1 手导致超风险）。

### 8.3 代码：资本与合约粒度



In [6]:

mult = np.round(100_000 / price0)                  # 教学示例：1 手名义约 10 万元
print("展期：carry 通过每月展期进入收益；忽略展期只做现货，年化差约等于平均 carry")
print("  全样本平均 |carry| =", round(float(np.abs(carry).mean()), 4))
print("\n资本与合约整数化（目标波动 10%，1 手名义约 10 万元）：")
w_day = W[600]
for cap in [100_000, 500_000, 2_000_000, 10_000_000]:
    desired = np.abs(w_day) * cap
    contract_notional = price[600] * mult
    lots = np.round(desired / contract_notional)
    actual = lots * contract_notional
    risk = np.sqrt(np.sum((actual * vol_36[600]) ** 2))
    print(f"  资本 {cap:>10,.0f}：平均持仓 {lots.sum():>3.0f} 手，"
          f"实际风险 {risk / cap:>7.2%}（目标 10%），零仓位数 {int((lots == 0).sum())}/8")


展期：carry 通过每月展期进入收益；忽略展期只做现货，年化差约等于平均 carry
  全样本平均 |carry| = 0.0319

资本与合约整数化（目标波动 10%，1 手名义约 10 万元）：
  资本    100,000：平均持仓   1 手，实际风险   5.94%（目标 10%），零仓位数 7/8
  资本    500,000：平均持仓   9 手，实际风险   9.84%（目标 10%），零仓位数 4/8
  资本  2,000,000：平均持仓  42 手，实际风险  10.92%（目标 10%），零仓位数 1/8
  资本 10,000,000：平均持仓 209 手，实际风险  10.83%（目标 10%），零仓位数 0/8



**读数与应用**：资本 10 万时，8 个品种里 7 个开不出仓位，实际风险只有 5.94%——不是“太保守”，而是**仓位不可控**；50 万时仍有 4 个品种零仓位；到 200 万以上才接近 10% 目标。这就是 Carver 说“散户需要足够资本”的真实含义：**不是不能开始，而是资金少时策略会被整数手和集中度扭曲。**

**常见错误**：

1. 用回测的名义仓位直接下单，不看整数手。
2. 资金少却硬上 8 个品种，每个品种 0/1 手，风险全押在少数品种。
3. 忽略保证金占用与强平线，杠杆算得过大。

---

### 8.4 执行、现金与复利

- **执行**：日频策略也要算滑点/冲击；用限价/算法单，记录“信号价 vs 成交价”的差，别假设收盘价成交。
- **现金**：保证金之外的现金有利息/机会成本；复利让资本曲线非线性，回测要按“可用资本”而不是“名义本金”算。
- **风险**：杠杆 = 名义敞口 / 资本；按波动率目标动态调整，不要固定手数、也不要亏了加仓。

## 本章小结

### 一张表

| 主题 | 一句话 | 本章证据 |
|---|---|---|
| 展期 | 总收益 = 现货变化 + carry | 8 品种分解表 |
| 波动率目标 | 先定风险，再定手数 | 目标 10%，实际 12.4% |
| 趋势 | 快慢多窗口各有取舍 | 慢趋势 Sharpe 1.13 > 快趋势 0.38 |
| carry | 赚期限结构的钱 | 仅 carry Sharpe 0.35 |
| 组合 | 低相关信号组合更稳 | 趋势+carry 波动降到 12.4% |
| 成本 | 换手高则成本致命 | 3bp→1.05，10bp→0.45，20bp→负 |
| 资本 | 整数手扭曲风险 | 10 万：7/8 品种零仓位 |
| 相对价值 | 高杠杆低容量 | 先方向性、后相对价值 |

### 八句话

1. 期货总收益 = 现货变化 + 展期收益，忽略展期回测会偏。
2. 先做波动率目标，再谈信号；风险管不好，信号再好也会爆。
3. 仓位与波动成反比，波动上升自动减仓。
4. 快趋势反应快但换手高，慢趋势更稳；多窗口平均是折中。
5. carry 和趋势逻辑不同、相关性低，组合起来更稳。
6. 日频期货策略的换手通常很高，成本必须认真算。
7. 资金少时整数手会让仓位和风险都失真。
8. 先把方向性策略做扎实，再碰均值回归和相对价值。

---

## 本章习题

**1.** 期货合约至少要看哪五个要素？为什么“连续合约”需要复权/展期处理？

**2.** 期货总收益可以拆成哪两部分？carry 为正时，对多头是收益还是成本？

**3.** 目标波动 10%、某品种实现波动 20%，它的风险缩放系数是多少？波动升到 40% 时仓位怎么变？

**4.** 快趋势和慢趋势各有什么优缺点？多窗口趋势为什么可能更稳？

**5.** carry 与趋势为什么能互补？组合后波动通常会怎样变化？

**6.** 某策略年换手 100 倍、单边成本 5bp，一年成本大约多少？对 Sharpe 的影响取决于什么？

**7.** 快速均值回归策略最大的风险是什么？相对价值策略（价差/三腿/日历）有什么额外风险？

**8.** 为什么资金 10 万时 8 个品种里 7 个开不出仓位？对一人量化，合理的策略建设顺序是什么？

<details><summary>参考答案</summary>

**1.** 标的（品种与交割月）、合约乘数、最小变动、保证金、到期/交割。连续合约要处理换月价差，否则价格序列有跳空，趋势与波动都会被污染。

**2.** 总收益 ≈ 现货价格变化 + 展期收益（carry）。carry 为正（近月贵/远月便宜）时，多头换月变便宜，是收益；carry 为负时是成本。

**3.** 缩放系数 = 10%/20% = 0.5；波动升到 40% 时缩放系数 = 0.25，仓位减半再减半。

**4.** 快趋势反应快但噪声大、换手高；慢趋势更稳但反应慢、可能错过拐点。多窗口平均把不同速度的信号混合，降低“单一速度失效”的风险。

**5.** 趋势赚价格延续的钱，carry 赚期限结构的钱，两者驱动不同、相关性低；组合后收益来源分散，波动通常下降。

**6.** 100 × 5bp = 500bp = 5%/年。对 Sharpe 的影响取决于策略波动：若波动 10%，成本约吃掉 0.5 的 Sharpe。

**7.** 快速均值回归单笔利润小、换手高，成本稍高就由正转负；相对价值杠杆高、容量小，且价差可能因结构变化“一去不回”。

**8.** 因为 1 手名义价值约 10 万，而 10 万资金按 10% 风险目标分到 8 个品种，每个品种的仓位不足 1 手。合理顺序：波动率目标与风险预算 → 趋势/carry 方向性策略 → 均值回归 → 相对价值。

</details>

---

## 参考资料

- Carver, R. *Advanced Futures Trading Strategies: 30 fully tested strategies for multiple trading styles and time frames*. Harriman House.（原书）
- Carver, R. *Systematic Trading*（波动率目标与风险预算的入门版）。
- Hurst, B., Ooi, Y. H., & Pedersen, L. H. *A Century of Evidence on Trend-Following Investing*.
- 本系列 `02_SmartBeta与因子投资_核心笔记`、`06_强化学习金融_核心笔记`。
